# Small classifier speed benchmark

This notebook measures graph-to-feature conversion separately from classifier fitting, then compares linear SGD, Random Forest, Extra Trees, and XGBoost on the same sparse matrix and homology-aware split. It uses existing local graph and sequence caches; it does not download protein data. The small-subset scores are diagnostic, not final model estimates.

Run from this repository checkout after notebooks 01–02 have created the active manifest, graph cache, and status file. MMseqs2 must be installed so the subset split can keep homologous sequences together.


In [ ]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "notebooks" / "src" / "enzyme_workflow").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the SimplexPairGraphVectorizer checkout")
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))

from enzyme_workflow.common import active_manifest, add_project_sources
add_project_sources(PROJECT_ROOT)


In [ ]:
# Keep the pilot small. Up to 100 proteins per EC class gives at most 700 rows at EC level 1.
MAX_PER_CLASS = 100
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_JOBS = -1
NODE_LABEL_ALPHABET = "aa20"
SIMPLEX_DIMENSION = (0, 3)
PAIR_DISTANCE = (1, 1)
NBITS = 11
FOREST_TREES = 200
XGB_ROUNDS = 300  # matches the current Notebook 03 setting


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer

from enzyme_workflow.classifier import (
    _fetch_uniprot_sequences,
    _grouped_train_test_split,
    _sequence_homology_groups,
)
from abstractgraph_graphicalizer.protein import ProteinLabelGraphicalizer, read_graph_pickle
from simplex_pair_vectorizer import SimplexPairGraphVectorizer


In [ ]:
manifest_path, ec_level = active_manifest(PROJECT_ROOT)
status_path = PROJECT_ROOT / "data" / "processed" / f"graph_build_status_level_{ec_level}.tsv"
if not status_path.is_file():
    raise FileNotFoundError(f"Run notebook 02 first; missing {status_path}")

manifest = pd.read_csv(manifest_path, sep="\t", dtype=str).fillna("")
status = pd.read_csv(status_path, sep="\t", dtype=str).fillna("")
dataset = manifest.merge(status, on="sample_id", how="inner", suffixes=("", "_build"))
dataset = dataset[dataset["status"].isin(["created", "skipped_existing"])].copy()
dataset = dataset[dataset["graph_path"].map(lambda value: Path(value).is_file())].reset_index(drop=True)
if dataset.empty:
    raise RuntimeError("No completed graph files found for the active manifest")

# Require the local sequence cache to keep this benchmark offline.
sequence_cache = PROJECT_ROOT / "data" / "interim" / "uniprot_sequences.tsv"
if not sequence_cache.is_file():
    raise FileNotFoundError(f"Offline benchmark needs the existing UniProt sequence cache: {sequence_cache}")
cached_sequences = pd.read_csv(sequence_cache, sep="\t", dtype=str).fillna("")
cached_accessions = set(cached_sequences["uniprot_accession"])
missing_accessions = sorted(set(dataset["uniprot_accession"]) - cached_accessions)
if missing_accessions:
    raise RuntimeError(f"Sequence cache is missing {len(missing_accessions)} accessions; populate it before running offline")

accessions = dataset["uniprot_accession"].astype(str)
sequences = _fetch_uniprot_sequences(accessions, sequence_cache)
homology_groups = _sequence_homology_groups(
    accessions,
    sequences,
    PROJECT_ROOT / "data" / "interim" / "sequence_homology",
    min_sequence_identity=0.30,
    min_coverage=0.80,
    n_jobs=N_JOBS,
)
dataset["homology_cluster"] = accessions.map(homology_groups)

rng_seeded_subset = []
for _, class_rows in dataset.groupby("ec_number", sort=True):
    rng_seeded_subset.append(
        class_rows.sample(n=min(MAX_PER_CLASS, len(class_rows)), random_state=RANDOM_STATE)
    )
subset = pd.concat(rng_seeded_subset).sort_values("sample_id").reset_index(drop=True)
y = subset["ec_number"].astype(str).to_numpy()
groups = subset["homology_cluster"].astype(str).to_numpy()
train_idx, test_idx, split_strategy = _grouped_train_test_split(
    y, groups, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"EC level {ec_level}; subset {len(subset):,} proteins across {subset['ec_number'].nunique()} classes")
print(f"Train {len(train_idx):,}; test {len(test_idx):,}; split: {split_strategy}")
display(subset.groupby("ec_number").size().rename("subset_samples").to_frame())


In [ ]:
graphs = [read_graph_pickle(path) for path in subset["graph_path"]]
label_graphicalizer = ProteinLabelGraphicalizer(alphabet=NODE_LABEL_ALPHABET)
start = time.perf_counter()
label_graphs = label_graphicalizer.fit_transform(graphs)
label_graphicalization_seconds = time.perf_counter() - start

vectorizer = SimplexPairGraphVectorizer(
    simplex_dimension=SIMPLEX_DIMENSION,
    pair_distance=PAIR_DISTANCE,
    nbits=NBITS,
    dense=False,
    include_self=True,
    n_jobs=N_JOBS,
    progress=True,
    clique_enumerator="auto",
)
start = time.perf_counter()
X = vectorizer.fit_transform(label_graphs)
vectorization_seconds = time.perf_counter() - start
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

matrix_bytes = X.data.nbytes + X.indices.nbytes + X.indptr.nbytes
density = X.nnz / (X.shape[0] * X.shape[1])
print(f"Label conversion: {label_graphicalization_seconds:.1f} s")
print(f"Simplex-pair vectorization: {vectorization_seconds:.1f} s")
print(f"CSR shape: {X.shape}; nnz: {X.nnz:,}; density: {density:.3%}; memory: {matrix_bytes / 2**20:.1f} MiB")


In [ ]:
from time import perf_counter
results = []

def benchmark(name, fit_model, predict_labels):
    start = perf_counter()
    fit_model()
    fit_seconds = perf_counter() - start
    start = perf_counter()
    prediction = predict_labels()
    predict_seconds = perf_counter() - start
    results.append({
        "model": name,
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        "accuracy": accuracy_score(y_test, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_test, prediction),
    })

linear = make_pipeline(
    Normalizer(norm="l2"),
    SGDClassifier(
        loss="hinge", alpha=1e-4, max_iter=5000, tol=1e-4,
        class_weight="balanced", average=True, random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
)
benchmark("Linear SGD", lambda: linear.fit(X_train, y_train), lambda: linear.predict(X_test))

forest = RandomForestClassifier(
    n_estimators=FOREST_TREES, max_features="sqrt", min_samples_leaf=2,
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
benchmark("Random Forest", lambda: forest.fit(X_train, y_train), lambda: forest.predict(X_test))

extra = ExtraTreesClassifier(
    n_estimators=FOREST_TREES, max_features="sqrt", min_samples_leaf=2,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=N_JOBS,
)
benchmark("Extra Trees", lambda: extra.fit(X_train, y_train), lambda: extra.predict(X_test))

from xgboost import XGBClassifier
xgb_classes = sorted(set(y_train))
class_to_id = {label: index for index, label in enumerate(xgb_classes)}
y_train_xgb = np.asarray([class_to_id[label] for label in y_train], dtype=np.int32)
class_frequencies = np.bincount(y_train_xgb, minlength=len(xgb_classes))
xgb_sample_weight = len(y_train_xgb) / (len(xgb_classes) * class_frequencies[y_train_xgb])
xgb = XGBClassifier(
    n_estimators=XGB_ROUNDS, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", n_jobs=N_JOBS, random_state=RANDOM_STATE, verbosity=0,
)
benchmark(
    "XGBoost",
    lambda: xgb.fit(X_train, y_train_xgb, sample_weight=xgb_sample_weight),
    lambda: np.asarray(xgb_classes, dtype=object)[xgb.predict(X_test).astype(int)],
)

timings = pd.DataFrame(results).set_index("model").sort_values("fit_seconds")
display(timings)


## Reading the result

Compare `fit_seconds` after the one-time vectorization timings above. The shared CSR matrix and shared homology-aware split make the classifier fit times directly comparable. The subset score is only a quick accuracy check; use Notebook 03 for a larger final evaluation.

XGBoost retains the current 300-round, depth-4 configuration. The forest pilot uses 200 trees to keep the first run short; increase `FOREST_TREES` if you want to compare at the current Random Forest setting of 500 trees. Rerun the final classifier cell to repeat model timings without rebuilding the feature matrix.
